In [1]:
import numpy as np
import math
import random
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# Define possible models and hyperparameters
models = {
    'DecisionTree': DecisionTreeClassifier,
    'SVM': SVC,
    'KNN': KNeighborsClassifier
}

hyperparameters = {
    'DecisionTree': {'max_depth': [3, 5, 10]},
    'SVM': {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']},
    'KNN': {'n_neighbors': [3, 5, 10]}
}

In [ ]:
# Objective function: negative cross-validation score
def objective_function(model, params, X, y):
    ml_model = model(**params)
    scores = cross_val_score(ml_model, X, y, cv=5, scoring='accuracy')
    return 1 - np.mean(scores)

In [3]:
# Simulated Annealing algorithm
def simulated_annealing(models, hyperparameters, X, y, t_max, cooling_rate, n_iterations):

    # randomly choose the model and params
    current_model = np.random.choice(list(models.keys()))
    current_params = {param: np.random.choice(values) for param, values in hyperparameters[current_model].items()}

    # counting score for this choice
    current_score = objective_function(models[current_model], current_params, X, y)

    # if new choice is the best - saving it
    best_model, best_params, best_score = current_model, current_params, current_score

    temperature = t_max

    for i in range(n_iterations):
        # Generate new candidate
        new_model = np.random.choice(list(models.keys()))
        new_params = {param: np.random.choice(values) for param, values in hyperparameters[new_model].items()}
        new_score = objective_function(models[new_model], new_params, X, y)

        # Calculating the acceptance probability
        delta = new_score - current_score
        if delta <= 0:
            acceptance_prob = 1
        else:
            acceptance_prob = np.exp(-delta / temperature)

        # If the acceptance probability is greater than the random number, the algorithm accepts the new solution
        if acceptance_prob > np.random.rand():
            current_model, current_params, current_score = new_model, new_params, new_score

        # Updating best solution
        if current_score < best_score:
            best_model, best_params, best_score = current_model, current_params, current_score

        # Cooling down
        # cooling rate, typically a value between 0.8 and 0.99
        # higher the rate slower the cooling, allowing more exploration
        temperature *= cooling_rate

    return best_model, best_params, best_score